# 1. Data Ingestion
- Read all CSV files using PySpark
- Ensure dynamic file handling (no hardcoding of file names)
- Add a new column: category_name

In [0]:
from pyspark.sql import functions as F
import os
import re
import sys

# Import configuration
sys.path.append("/Workspace/Users/suthar2406@gmail.com/data_engineering_tasks/E_Commerce_Task")
from config import Config

print("STEP 1: DATA INGESTION")

try:
    # List files from configured source directory
    print(f"\n[INFO] Reading files from: {Config.SOURCE_DIRECTORY}")
    files = dbutils.fs.ls(Config.SOURCE_DIRECTORY)
    csv_files = [f.path for f in files if f.path.endswith(Config.FILE_EXTENSION)]
    
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {Config.SOURCE_DIRECTORY}")
    
    print(f"[INFO] Found {len(csv_files)} CSV files to process")
    
except Exception as e:
    print(f"[ERROR] Failed to list files from source directory: {str(e)}")
    raise

dataframes = []
failed_files = []

for idx, file_path in enumerate(csv_files, 1):
    try:
        print(f"\n[{idx}/{len(csv_files)}] Processing: {os.path.basename(file_path)}")
        
        # Extract category name from filename
        base_name = os.path.basename(file_path).replace(Config.FILE_EXTENSION, "")
        parts = base_name.split('-')
        
        # Remove prefix (us-shein) and suffix (numeric ID)
        category_parts = parts[Config.FILENAME_PREFIX_PARTS_TO_REMOVE:-Config.FILENAME_SUFFIX_PARTS_TO_REMOVE]
        clean_cat = "_".join(category_parts) if category_parts else base_name
        
        print(f"    Extracted category: '{clean_cat}'")
        
        # Read CSV with schema inference
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(file_path)
        
        print(f"    Loaded {df.count()} records with {len(df.columns)} columns")
        
        # Standardize column names
        for col_name in df.columns:
            clean_col = re.sub(r'[^a-zA-Z0-9]', '_', col_name).lower()
            df = df.withColumnRenamed(col_name, clean_col)
        
        # Add category identifier
        df = df.withColumn("category_name", F.lit(clean_cat))
        dataframes.append(df)
        
        print(f"    Successfully processed")
        
    except Exception as e:
        print(f"    FAILED: {str(e)}")
        failed_files.append((file_path, str(e)))
        continue

print("\nINGESTION SUMMARY")
print(f"Total files found: {len(csv_files)}")
print(f"Successfully loaded: {len(dataframes)}")
print(f"Failed: {len(failed_files)}")

if failed_files:
    print("\n[WARNING] Failed files:")
    for file_path, error in failed_files:
        print(f"  - {os.path.basename(file_path)}: {error}")

if not dataframes:
    raise RuntimeError("No dataframes were successfully loaded. Pipeline cannot continue.")

print(f"\n[SUCCESS] Ingestion complete. {len(dataframes)} categories ready for processing.")

# 2. Data Cleaning
- Standardize column names
- Handle missing/null values
- Handle columns that exist in some files but not others
- Remove unnecessary columns (if identified)

In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

print("\nSTEP 2: DATA CLEANING")

def standardize_names(df):
    # Standardize column names to lowercase with underscores
    try:
        for col in df.columns:
            clean_name = re.sub(r'[^a-zA-Z0-9]', '_', col).lower().strip('_')
            if clean_name != col:
                df = df.withColumnRenamed(col, clean_name)
        return df
    except Exception as e:
        raise ValueError(f"Failed to standardize column names: {str(e)}")

try:
    print(f"\n[INFO] Standardizing column names for {len(dataframes)} dataframes...")
    
    cleaned_count = 0
    for idx, df in enumerate(dataframes, 1):
        try:
            category = df.select("category_name").first()[0]
            print(f"[{idx}/{len(dataframes)}] Cleaning: {category}")
            
            original_cols = len(df.columns)
            dataframes[idx-1] = standardize_names(df)
            cleaned_count += 1
            
            print(f"    Standardized {original_cols} column names")
            
        except Exception as e:
            print(f"    Failed to clean {category}: {str(e)}")
            raise
    
    # Preserve nulls to maintain data type integrity
    print(f"\n[INFO] Preserving null values to maintain data type integrity")
    final_dfs = dataframes
    
    print("\nCLEANING SUMMARY")
    print(f"Dataframes cleaned: {cleaned_count}/{len(dataframes)}")
    print(f"Null handling: Preserved (will be handled in final cleanup)")
    print(f"\n[SUCCESS] Data cleaning complete.")
    
except Exception as e:
    print(f"\n[ERROR] Data cleaning failed: {str(e)}")
    raise

# 3. Data Transformation
- Convert string values to appropriate data types:
  - Remove $ from price
  - Remove % from discount
  - Convert "k" values (e.g., 5k → 5000)
- Create standardized columns:
  - price_usd
  - pct_discount
  - qty_sold
- Extract numeric rank from values like #1, #2

In [0]:
from pyspark.sql import functions as F
import sys

# Import configuration
sys.path.append("/Workspace/Users/suthar2406@gmail.com/data_engineering_tasks/E_Commerce_Task")
from config import Config

print("\nSTEP 3: DATA TRANSFORMATIONS")

def apply_safe_transformations(df):
    # Apply business logic transformations
    try:
        # Product columns transformation
        if "goods_title_link" in df.columns or "goods_title_link__jump" in df.columns:
            df = df.withColumn("product_description", 
                F.coalesce(
                    F.col("goods_title_link") if "goods_title_link" in df.columns else F.lit(None),
                    F.col("goods_title_link__jump") if "goods_title_link__jump" in df.columns else F.lit(None)
                )
            )
            
            # Create shortened product name
            cleaned = F.regexp_replace(
                F.col("product_description"), 
                Config.QUANTITY_PREFIX_PATTERN,
                ""
            )
            
            df = df.withColumn("product_name",
                F.when(
                    cleaned.contains(","),
                    F.regexp_replace(
                        F.split(cleaned, ",").getItem(0),
                        Config.FILLER_WORDS_PATTERN,
                        ""
                    )
                ).otherwise(
                    F.substring(
                        F.regexp_replace(
                            cleaned,
                            Config.FILLER_WORDS_PATTERN,
                            ""
                        ),
                        1, Config.MAX_PRODUCT_NAME_LENGTH
                    )
                )
            )
            
            df = df.withColumn("product_name", F.trim(F.col("product_name")))
        
        # Product URL
        if "goods_title_link__jump_href" in df.columns:
            df = df.withColumn("product_url", F.col("goods_title_link__jump_href"))
        
        # Rank category transformation
        if "rank_sub" in df.columns:
            df = df.withColumn("rank_category", 
                F.when(F.col("rank_sub").isNull(), F.lit("unknown"))
                 .otherwise(F.regexp_replace(F.col("rank_sub"), Config.RANK_PREFIX_PATTERN, ""))
            )
        
        # Price conversion (USD)
        if "price" in df.columns:
            cleaned = F.regexp_replace(F.col("price"), r"[\$,]", "")
            df = df.withColumn("price_usd", 
                F.when(cleaned.rlike(r"^\d*\.?\d+$"), cleaned.cast("double"))
                 .otherwise(F.lit(None))
            )
        
        # Discount percentage conversion
        if "discount" in df.columns:
            cleaned = F.regexp_replace(F.col("discount"), r"[% \s-]", "")
            df = df.withColumn("pct_discount", 
                F.when(cleaned.rlike(r"^\d+$"), cleaned.cast("int"))
                 .otherwise(F.lit(None))
            )
        
        # Quantity sold extraction (handle 'k' multiplier)
        if "selling_proposition" in df.columns:
            num_str = F.regexp_extract(F.col("selling_proposition"), r"(\d+\.?\d*)", 1)
            valid_num = F.when(num_str.rlike(r"^\d*\.?\d+$"), num_str.cast("double")).otherwise(F.lit(None))
            
            df = df.withColumn("qty_sold", 
                F.when(F.col("selling_proposition").rlike("(?i)k"), valid_num * 1000)
                 .otherwise(valid_num)
            )
        
        # Rank number extraction
        if "rank_title" in df.columns:
            rank_str = F.regexp_extract(F.col("rank_title"), r"#(\d+)", 1)
            df = df.withColumn("rank_number", 
                F.when(rank_str.rlike(r"^\d+$"), rank_str.cast("int"))
                 .otherwise(F.lit(None))
            )

        # Drop original and unnecessary columns
        df = df.drop(*[c for c in Config.COLUMNS_TO_DROP if c in df.columns])
        
        return df
        
    except Exception as e:
        raise RuntimeError(f"Transformation failed: {str(e)}")

try:
    print(f"\n[INFO] Applying transformations to {len(final_dfs)} dataframes...")
    
    transformed_dfs = []
    for idx, df in enumerate(final_dfs, 1):
        try:
            category = df.select("category_name").first()[0]
            print(f"\n[{idx}/{len(final_dfs)}] Transforming: {category}")
            
            original_cols = len(df.columns)
            transformed_df = apply_safe_transformations(df)
            transformed_cols = len(transformed_df.columns)
            
            transformed_dfs.append(transformed_df)
            
            print(f"    Columns: {original_cols} -> {transformed_cols}")
            print(f"    Transformations applied successfully")
            
        except Exception as e:
            print(f"    Failed: {str(e)}")
            raise
    
    print("\nTRANSFORMATION SUMMARY")
    print(f"Dataframes transformed: {len(transformed_dfs)}/{len(final_dfs)}")
    print(f"New columns created: product_description, product_name, product_url,")
    print(f"                     price_usd, pct_discount, qty_sold, rank_number, rank_category")
    print(f"Columns dropped: {len(Config.COLUMNS_TO_DROP)}")
    print(f"\n[SUCCESS] Data transformation complete.")
    
except Exception as e:
    print(f"\n[ERROR] Transformation failed: {str(e)}")
    raise

# 4. Data Merging
- Combine all category DataFrames into one
- Ensure schema consistency across all datasets

In [0]:
from functools import reduce
from pyspark.sql import functions as F
import sys

# Import configuration
sys.path.append("/Workspace/Users/suthar2406@gmail.com/data_engineering_tasks/E_Commerce_Task")
from config import Config

print("\nSTEP 4: SCHEMA-ALIGNED DATA MERGING")

try:
    # Collect all unique column names
    print(f"\n[INFO] Analyzing schemas across {len(transformed_dfs)} dataframes...")
    
    all_columns = set()
    for df in transformed_dfs:
        all_columns.update(df.columns)
    
    print(f"[INFO] Found {len(all_columns)} unique columns across all categories")
    print(f"[INFO] Numeric columns to preserve: {Config.NUMERIC_COLUMNS}")
    
    # Align schemas by casting non-numeric columns to string
    print(f"\n[INFO] Aligning schemas for safe union...")
    aligned_dfs = []
    
    for idx, df in enumerate(transformed_dfs, 1):
        try:
            category = df.select("category_name").first()[0]
            print(f"[{idx}/{len(transformed_dfs)}] Aligning: {category}")
            
            casted_count = 0
            for col in df.columns:
                if col not in Config.NUMERIC_COLUMNS and col in df.columns:
                    df = df.withColumn(col, F.col(col).cast("string"))
                    casted_count += 1
            
            aligned_dfs.append(df)
            print(f"    Casted {casted_count} columns to string")
            
        except Exception as e:
            print(f"    Failed to align {category}: {str(e)}")
            raise
    
    print(f"\n[INFO] Performing union operation...")
    
    # Union all dataframes
    master_df = reduce(
        lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), 
        aligned_dfs
    )
    
    # Validate merge
    total_records = master_df.count()
    total_columns = len(master_df.columns)
    
    if total_records == 0:
        raise ValueError("Union resulted in empty dataframe")
    
    print("\nMERGE SUMMARY")
    print(f"Categories merged: {len(transformed_dfs)}")
    print(f"Total records: {total_records:,}")
    print(f"Total columns: {total_columns}")
    print(f"\n[SUCCESS] Data merge complete.")
    
except Exception as e:
    print(f"\n[ERROR] Data merge failed: {str(e)}")
    raise

# 5. Data Quality Checks
- Identify duplicate records
- Remove or handle duplicates appropriately
- Validate null values in key columns
- Validate data types after transformation

In [0]:
from pyspark.sql import functions as F
import sys

# Import configuration
sys.path.append("/Workspace/Users/suthar2406@gmail.com/data_engineering_tasks/E_Commerce_Task")
from config import Config

print("\nSTEP 5: FINAL CLEANUP")

try:
    initial_count = master_df.count()
    print(f"\n[INFO] Starting cleanup with {initial_count:,} records")
    
    # Deduplication
    print(f"\n[INFO] Removing duplicates...")
    final_df = master_df.dropDuplicates()
    
    post_dedup_count = final_df.count()
    duplicates_removed = initial_count - post_dedup_count
    
    if duplicates_removed > 0:
        print(f"[INFO] Removed {duplicates_removed:,} duplicate records")
    else:
        print(f"[INFO] No duplicates found")
    
    # Fill numeric nulls with configured values
    print(f"\n[INFO] Filling numeric nulls...")
    
    fillna_dict = {}
    for col, fill_value in Config.NUMERIC_FILLNA_COLUMNS.items():
        if col in final_df.columns:
            fillna_dict[col] = fill_value
    
    if fillna_dict:
        final_df = final_df.fillna(fillna_dict)
        print(f"[INFO] Filled nulls in {len(fillna_dict)} numeric columns")
        for col, val in fillna_dict.items():
            print(f"    {col}: {val}")
    
    # Final validation
    final_count = final_df.count()
    
    if final_count == 0:
        raise ValueError("Final dataframe is empty after cleanup")
    
    print("\nFINAL CLEANUP SUMMARY")
    print(f"Initial merged records: {initial_count:,}")
    print(f"Duplicates removed: {duplicates_removed:,}")
    print(f"Final clean records: {final_count:,}")
    print(f"Total columns: {len(final_df.columns)}")
    print(f"\n[SUCCESS] Pipeline complete!")
    
    # Display sample
    print(f"\n[INFO] Displaying sample of {Config.SAMPLE_SIZE_FOR_DISPLAY} records:\n")
    display(final_df.limit(Config.SAMPLE_SIZE_FOR_DISPLAY))
    
    # Schema display
    print("\n[INFO] Final schema:")
    final_df.printSchema()
    
except Exception as e:
    print(f"\n[ERROR] Final cleanup failed: {str(e)}")
    raise

# 6. Data Output
- Save final cleaned dataset to CSV format
- Create output directory alongside source data
- Optionally save as Parquet for better performance

In [0]:
import sys
import os

# Import configuration
sys.path.append("/Workspace/Users/suthar2406@gmail.com/data_engineering_tasks/E_Commerce_Task")
from config import Config

print("\nSTEP 6: DATA OUTPUT")

try:
    # Determine output directory path
    source_dir = Config.SOURCE_DIRECTORY.rstrip('/')
    parent_dir = os.path.dirname(source_dir)
    output_dir = os.path.join(parent_dir, "output")
    
    print(f"\n[INFO] Source directory: {source_dir}")
    print(f"[INFO] Output directory: {output_dir}")
    
    # Create output directory if it doesn't exist
    try:
        dbutils.fs.mkdirs(output_dir)
        print(f"[INFO] Output directory ready")
    except Exception as e:
        print(f"[WARNING] Directory may already exist: {str(e)}")
    
    # Define output file paths
    csv_output_path = os.path.join(output_dir, "us_shein_products_cleaned.csv")
    parquet_output_path = os.path.join(output_dir, "us_shein_products_cleaned.parquet")
    
    # Save as CSV
    print(f"\n[INFO] Saving dataset as CSV...")
    final_df.coalesce(1) \
        .write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(csv_output_path)
    
    print(f"[SUCCESS] CSV saved to: {csv_output_path}")
    
    # Save as Parquet
    print(f"\n[INFO] Saving dataset as Parquet...")
    final_df.write \
        .mode("overwrite") \
        .parquet(parquet_output_path)
    
    print(f"[SUCCESS] Parquet saved to: {parquet_output_path}")
    
    # Display output summary
    record_count = final_df.count()
    column_count = len(final_df.columns)
    
    print("\nOUTPUT SUMMARY")
    print(f"Records saved: {record_count:,}")
    print(f"Columns saved: {column_count}")
    print(f"\nOutput formats:")
    print(f"  1. CSV:     {csv_output_path}")
    print(f"  2. Parquet: {parquet_output_path}")
    print(f"\n[SUCCESS] Data pipeline complete! Files ready for use.")
    
except Exception as e:
    print(f"\n[ERROR] Failed to save output: {str(e)}")
    raise

In [0]:
import sys
import os

# Import configuration
sys.path.append("/Workspace/Users/suthar2406@gmail.com/data_engineering_tasks/E_Commerce_Task")
from config import Config

print("STEP 7: OUTPUT VERIFICATION")
print("\n[INFO] Loading saved files to verify integrity...")

try:
    # Define paths
    source_dir = Config.SOURCE_DIRECTORY.rstrip('/')
    parent_dir = os.path.dirname(source_dir)
    output_dir = os.path.join(parent_dir, "output")
    
    csv_path = os.path.join(output_dir, "us_shein_products_cleaned.csv")
    parquet_path = os.path.join(output_dir, "us_shein_products_cleaned.parquet")
    
    # Load CSV file
    print("\n[INFO] Reading CSV file...")
    csv_df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(csv_path)
    
    csv_rows = csv_df.count()
    csv_cols = len(csv_df.columns)
    
    print(f"    CSV rows: {csv_rows:,}")
    print(f"    CSV columns: {csv_cols}")
    
    # Load Parquet file
    print("\n[INFO] Reading Parquet file...")
    parquet_df = spark.read.parquet(parquet_path)
    
    parquet_rows = parquet_df.count()
    parquet_cols = len(parquet_df.columns)
    
    print(f"    Parquet rows: {parquet_rows:,}")
    print(f"    Parquet columns: {parquet_cols}")
    
    # Verification
    print("\nVERIFICATION RESULTS")
    
    expected_rows = 79242
    expected_cols = 10
    
    # Check row counts
    if csv_rows == expected_rows:
        print(f"[PASS] CSV has correct row count: {csv_rows:,}")
    else:
        print(f"[FAIL] CSV row count mismatch: expected {expected_rows:,}, got {csv_rows:,}")
    
    if parquet_rows == expected_rows:
        print(f"[PASS] Parquet has correct row count: {parquet_rows:,}")
    else:
        print(f"[FAIL] Parquet row count mismatch: expected {expected_rows:,}, got {parquet_rows:,}")
    
    # Check column counts
    if csv_cols == expected_cols:
        print(f"[PASS] CSV has correct column count: {csv_cols}")
    else:
        print(f"[FAIL] CSV column count mismatch: expected {expected_cols}, got {csv_cols}")
    
    if parquet_cols == expected_cols:
        print(f"[PASS] Parquet has correct column count: {parquet_cols}")
    else:
        print(f"[FAIL] Parquet column count mismatch: expected {expected_cols}, got {parquet_cols}")
    
    # Check if both formats match
    if csv_rows == parquet_rows:
        print(f"[PASS] CSV and Parquet row counts match")
    else:
        print(f"[FAIL] CSV and Parquet row counts differ: CSV={csv_rows:,}, Parquet={parquet_rows:,}")
    
    if csv_cols == parquet_cols:
        print(f"[PASS] CSV and Parquet column counts match")
    else:
        print(f"[FAIL] CSV and Parquet column counts differ: CSV={csv_cols}, Parquet={parquet_cols}")
    
    # Display column names
    print("\n[INFO] Column names:")
    for idx, col in enumerate(csv_df.columns, 1):
        print(f"    {idx}. {col}")
    
    # Overall status
    all_passed = (csv_rows == expected_rows and 
                  parquet_rows == expected_rows and 
                  csv_cols == expected_cols and 
                  parquet_cols == expected_cols and
                  csv_rows == parquet_rows and
                  csv_cols == parquet_cols)
    
    if all_passed:
        print("\n[SUCCESS] All verification checks passed!")
    else:
        print("\n[WARNING] Some verification checks failed. Review results above.")
    
except Exception as e:
    print(f"\n[ERROR] Verification failed: {str(e)}")
    raise